In [1]:
# Feature Engineering
import sys, pathlib, yaml
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

sys.path.append(str(pathlib.Path.cwd().parent))
from src.features import add_derived_features, fit_target_encoding, apply_target_encoding, prepare_model_frame
from src.model import fit_baseline, predict_baseline, train_point_model, train_quantile_model, evaluate, evaluate_by_price_segment, fit_conformal_margin, apply_conformal_margin
from src.comparables import find_comparables

pd.set_option("display.width", 120)

with open("../config/config.yaml") as f:
    cfg = yaml.safe_load(f)



In [2]:
REFERENCE_YEAR = 2026 

df = pd.read_csv("../" + cfg["paths"]["normalized"])
df["log_price"] = np.log(df["asking_price"])
print("rows:", len(df))

rows: 1626


In [3]:
# Derived features (no target/price involved -- safe to compute
# once on the full dataframe, before splitting)
df = add_derived_features(df, REFERENCE_YEAR)

In [4]:
# Train / test split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print(f"train: {len(train_df)}  test: {len(test_df)}")

train: 1300  test: 326


In [5]:
# Target-dependent features: fit on TRAIN ONLY, apply to both
#
# Target encoding and the group-median baseline both use price, so
# fitting them must happen strictly after the split -- fitting on the
# full dataframe first would leak each test row's own price into the
# feature used to predict it, inflating the test metrics below.

brand_model_enc_map = fit_target_encoding(train_df, "brand_model", "log_price", smoothing=10.0)
train_df = train_df.copy()
test_df = test_df.copy()
train_df["brand_model_target_enc"] = apply_target_encoding(train_df, "brand_model", brand_model_enc_map)
test_df["brand_model_target_enc"] = apply_target_encoding(test_df, "brand_model", brand_model_enc_map)

train_df = prepare_model_frame(train_df)
test_df = prepare_model_frame(test_df)

NUMERIC_FEATURES = [
    "car_age", "mileage_km", "mileage_per_year", "insurance_months_left",
    "brand_freq", "brand_model_target_enc", "is_pickup",
]
CATEGORICAL_FEATURES = ["brand", "city", "fuel_type", "Gearbox_type", "Gearbox_condition",
                         "engine_condition", "chassis_condition", "body_condition"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

for col in NUMERIC_FEATURES:
    train_df[col] = train_df[col].fillna(train_df[col].median())
    test_df[col] = test_df[col].fillna(train_df[col].median())  # fill with TRAIN median, not test's own

X_train, y_train = train_df[FEATURES], train_df["log_price"]
X_test, y_test = test_df[FEATURES], test_df["log_price"]

In [ ]:
# Baseline: hierarchical group median
baseline = fit_baseline(train_df)
baseline_pred = predict_baseline(test_df, baseline)
baseline_metrics = evaluate(test_df["asking_price"].values, baseline_pred)
print("Baseline (group median) metrics:")
for k, v in baseline_metrics.items():
    print(f"  {k}: {v:.2f}")

Baseline (group median) metrics:
  MAE: 758279141.10
  RMSE: 3354442358.55
  MAPE: 31.52
  median_APE: 14.90
  within_10pct: 36.50
  within_20pct: 61.04


In [ ]:
# Point-estimate model: LightGBM on log(price)
point_model = train_point_model(X_train, y_train, CATEGORICAL_FEATURES)

pred_log = point_model.predict(X_test)
pred_price = np.exp(pred_log)
gbm_metrics = evaluate(test_df["asking_price"].values, pred_price)
print("LightGBM point model metrics:")
for k, v in gbm_metrics.items():
    print(f"  {k}: {v:.2f}")

print()
print("Improvement over baseline (MAPE): "
      f"{baseline_metrics['MAPE']:.1f}% -> {gbm_metrics['MAPE']:.1f}%")

LightGBM point model metrics:
  MAE: 1011066063.47
  RMSE: 4127062892.37
  MAPE: 22.83
  median_APE: 13.22
  within_10pct: 38.34
  within_20pct: 65.64

Improvement over baseline (MAPE): 31.5% -> 22.8%


In [8]:
# Error by price segment
#
# Aggregate MAE/RMSE/MAPE above are dominated by the priciest quartile
# (a handful of luxury/rare imports with a few Toman-billions of
# absolute error each). This breakdown shows the model is much more
# reliable in the common/mainstream price range than the aggregate
# numbers alone suggest -- report both in the final writeup.

segment_report = evaluate_by_price_segment(test_df["asking_price"].values, pred_price)
print(segment_report.to_string(index=False))

segment  n                    price_range          MAE         RMSE      MAPE  median_APE  within_10pct  within_20pct
     Q1 83      215,000,000 - 760,000,000 1.110986e+08 1.751361e+08 22.781143   12.827589     36.144578     69.879518
     Q2 80    770,000,000 - 1,350,000,000 2.006034e+08 3.031844e+08 19.745341   12.886330     42.500000     68.750000
     Q3 81  1,355,000,000 - 2,400,000,000 3.079323e+08 4.790601e+08 16.243993   10.787244     45.679012     72.839506
     Q4 82 2,425,000,000 - 55,000,000,000 3.407263e+09 8.207785e+09 32.375909   19.011589     29.268293     51.219512


In [ ]:
# Quantile models -- price range / uncertainty
# 10th and 90th percentile of log(price) -> an 80% prediction interval on
# the original price scale.
q_low_model = train_quantile_model(X_train, y_train, CATEGORICAL_FEATURES, alpha=0.10)
q_high_model = train_quantile_model(X_train, y_train, CATEGORICAL_FEATURES, alpha=0.90)

pred_low = np.exp(q_low_model.predict(X_test))
pred_high = np.exp(q_high_model.predict(X_test))

coverage = np.mean((test_df["asking_price"].values >= pred_low) & (test_df["asking_price"].values <= pred_high))
print(f"80% interval empirical coverage on test set: {coverage * 100:.1f}%  (target: 80%)")
print(f"median relative interval width: {np.median((pred_high - pred_low) / pred_price) * 100:.1f}%")

80% interval empirical coverage on test set: 68.4%  (target: 80%)
median relative interval width: 43.8%


In [ ]:
# Conformal calibration of the interval
#
# The 67.5% empirical coverage above (vs. the 80% target) means the
# quantile models are overconfident -- the interval is narrower than it
# should be. Conformalized Quantile Regression (CQR) fixes this using a
# held-out CALIBRATION split (carved out of train_df, never seen by the
# quantile models, and separate from test_df): measure how far off the
# quantile predictions actually are on that split, and widen the
# interval by exactly the margin needed to hit the target coverage.
  
model_train_df, calib_df = train_test_split(train_df, test_size=0.2, random_state=7)
 
# Re-fit target encoding on model_train_df ONLY (not the full train_df,
# which includes calib_df) -- otherwise calib_df's own price leaks into
# its brand_model_target_enc feature, undermining the calibration.
calib_enc_map = fit_target_encoding(model_train_df, "brand_model", "log_price", smoothing=10.0)
model_train_df = model_train_df.copy()
calib_df = calib_df.copy()
model_train_df["brand_model_target_enc"] = apply_target_encoding(model_train_df, "brand_model", calib_enc_map)
calib_df["brand_model_target_enc"] = apply_target_encoding(calib_df, "brand_model", calib_enc_map)
test_calib_features = test_df.copy()
test_calib_features["brand_model_target_enc"] = apply_target_encoding(test_calib_features, "brand_model", calib_enc_map)
 
q_low_model_c = train_quantile_model(model_train_df[FEATURES], model_train_df["log_price"], CATEGORICAL_FEATURES, alpha=0.10)
q_high_model_c = train_quantile_model(model_train_df[FEATURES], model_train_df["log_price"], CATEGORICAL_FEATURES, alpha=0.90)
 
calib_low_log = q_low_model_c.predict(calib_df[FEATURES])
calib_high_log = q_high_model_c.predict(calib_df[FEATURES])
margin = fit_conformal_margin(calib_df["log_price"].values, calib_low_log, calib_high_log, target_coverage=0.8)
print(f"conformal margin (log-price scale): {margin:.4f}")
 
test_low_log = q_low_model_c.predict(test_calib_features[FEATURES])
test_high_log = q_high_model_c.predict(test_calib_features[FEATURES])
cal_low_log, cal_high_log = apply_conformal_margin(test_low_log, test_high_log, margin)
cal_pred_low, cal_pred_high = np.exp(cal_low_log), np.exp(cal_high_log)
 
cal_coverage = np.mean((test_df["asking_price"].values >= cal_pred_low) & (test_df["asking_price"].values <= cal_pred_high))
print(f"80% interval empirical coverage AFTER calibration: {cal_coverage * 100:.1f}%  (target: 80%)")
print(f"median relative interval width AFTER calibration: {np.median((cal_pred_high - cal_pred_low) / pred_price) * 100:.1f}%")
 
# Final quantile models for deployment
#
# The margin above was learned on a calibration split so it reflects
# genuine held-out miscalibration, not something fit to the full
# training data. q_low_model/q_high_model from section 5 were already
# trained on all of train_df with the original brand_model_enc_map, so
# they're reused directly here -- no need to retrain an identical model.
# The margin is carried forward as-is at inference time (applied inside
# estimate_price via conformal_margin).

conformal margin (log-price scale): 0.1400
80% interval empirical coverage AFTER calibration: 80.4%  (target: 80%)
median relative interval width AFTER calibration: 68.3%


In [ ]:
# 5-fold cross-validation sanity check

# Confirms the single 80/20 split isn't a lucky/unlucky draw, given the
# small dataset size.
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_mapes = []
for fold_i, (tr_idx, va_idx) in enumerate(kf.split(df)):
    tr, va = df.iloc[tr_idx].copy(), df.iloc[va_idx].copy()
    tr = add_derived_features(tr, REFERENCE_YEAR)
    va = add_derived_features(va, REFERENCE_YEAR)
    enc_map = fit_target_encoding(tr, "brand_model", "log_price", smoothing=10.0)
    tr["brand_model_target_enc"] = apply_target_encoding(tr, "brand_model", enc_map)
    va["brand_model_target_enc"] = apply_target_encoding(va, "brand_model", enc_map)
    tr, va = prepare_model_frame(tr), prepare_model_frame(va)
    for col in NUMERIC_FEATURES:
        tr[col] = tr[col].fillna(tr[col].median())
        va[col] = va[col].fillna(tr[col].median())
    m = train_point_model(tr[FEATURES], tr["log_price"], CATEGORICAL_FEATURES)
    pred = np.exp(m.predict(va[FEATURES]))
    fold_mapes.append(evaluate(va["asking_price"].values, pred)["MAPE"])

print("5-fold MAPE:", [f"{m:.1f}%" for m in fold_mapes])
print(f"mean +/- std: {np.mean(fold_mapes):.1f}% +/- {np.std(fold_mapes):.1f}%")

5-fold MAPE: ['22.8%', '32.4%', '30.1%', '27.2%', '29.0%']
mean +/- std: 28.3% +/- 3.2%


In [ ]:
# Feature importance 
importance = pd.Series(point_model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print(importance)

brand_model_target_enc    2778
mileage_per_year          1866
mileage_km                1802
car_age                   1460
brand_freq                1177
insurance_months_left      916
brand                      440
engine_condition           346
chassis_condition          275
Gearbox_type               240
body_condition             229
Gearbox_condition          212
city                       204
fuel_type                   55
is_pickup                    0
dtype: int32


In [ ]:
# Save artifacts + call the standalone API 
 
from src.api import save_artifacts, load_artifacts, estimate_price as estimate_price_api
 
ARTIFACTS_DIR = "../artifacts"
save_artifacts(
    ARTIFACTS_DIR,
    point_model=point_model,
    q_low_model=q_low_model,
    q_high_model=q_high_model,
    brand_model_enc_map=brand_model_enc_map,
    conformal_margin=margin,
    full_df=df,
    reference_year=REFERENCE_YEAR,
)
artifacts = load_artifacts(ARTIFACTS_DIR)  
print(f"artifacts saved to and reloaded from {ARTIFACTS_DIR}/artifacts.pkl")


artifacts saved to and reloaded from ../artifacts/artifacts.pkl


In [16]:
# Five real example outputs 

sample_cars = test_df.sample(5, random_state=1)
for _, r in sample_cars.iterrows():
    car = {
        "brand_model": r["brand_model"], "brand": r["brand"], "city": r["city"],
        "year_gregorian": r["year_gregorian"], "mileage_km": r["mileage_km"],
        "fuel_type": r["fuel_type"], "Gearbox_type": r["Gearbox_type"],
        "Gearbox_condition": r["Gearbox_condition"], "engine_condition": r["engine_condition"],
        "chassis_condition": r["chassis_condition"], "body_condition": r["body_condition"],
        "is_pickup": r["is_pickup"], "insurance_months_left": r["insurance_months_left"],
        "asking_price": r["asking_price"],
    }
    out = estimate_price_api(car, artifacts)  # the standalone, imported function
    print(f"\n{r['brand_model']}  (actual asking: {r['asking_price']:,.0f})")
    print(f"  estimated: {out['estimated_price']:,}  range: {out['price_range']}  "
          f"position: {out.get('market_position')}  confidence: {out['confidence']}")
    print(f"  comparables used: {len(out['comparable_vehicles'])}")


پراید وانت 151 SE  (actual asking: 645,000,000)
  estimated: 614,562,528  range: (292434038, 868758311)  position: Fair  confidence: Medium
  comparables used: 3

ام وی ام 110 اتوماتیک ۴ سیلندر  (actual asking: 390,000,000)
  estimated: 604,894,112  range: (350690004, 1607136706)  position: Fair  confidence: Low
  comparables used: 5

تارا v1 پلاس  (actual asking: 2,050,000,000)
  estimated: 1,809,756,340  range: (1274709483, 2111242484)  position: Fair  confidence: Medium
  comparables used: 5

سایپا اطلس G  (actual asking: 1,470,000,000)
  estimated: 3,187,060,510  range: (1364381287, 5504387113)  position: Fair  confidence: Low
  comparables used: 5

جیلی Emgrand 7 دنده ای  (actual asking: 1,350,000,000)
  estimated: 1,401,795,138  range: (1158717049, 2037363498)  position: Fair  confidence: Medium
  comparables used: 3
